In [ ]:
import torch
from scope_diffuser import SCoPEDiffusionPipeline
import matplotlib.pyplot as plt
import random
import numpy as np

# Callback function to save intermediate steps
def save_intermediate(step, timestep, latents):
    with torch.no_grad():
        latents = 1 / 0.18215 * latents
        image = pipe.vae.decode(latents).sample
        image = (image / 2 + 0.5).clamp(0, 1)
        image = image.cpu().permute(0, 2, 3, 1).float().numpy()[0]
        intermediate_images.append(image)

num_inference_steps = 200
step_size = 10


for temperature in [1,10,100,1000]:
   
    # Load the model
    model_id = "CompVis/stable-diffusion-v1-4"
    pipe = SCoPEDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True)
    pipe = pipe.to("cuda:1")

    # Initialize list to store intermediate images
    intermediate_images = []

    # for scope diffusion
    prompt_schedule = [
            (0, "A waterfall made of glowing stars, pouring from the sky"),
            (step_size, "A waterfall made of glowing stars, cascading down from a break in the sky"),
            (step_size * 2, "A glowing waterfall of stars, pouring down from the sky and lighting up the landscape below"),
            (step_size * 3, "A glowing star waterfall, pouring from the sky, lighting up the landscape, and mist rising with sparkling particles"),
            (step_size * 4, "A waterfall of stars pouring from the sky, lighting the landscape, with mist filled with sparkling particles, and constellations forming in the mist")
    ]

    torch.manual_seed(42)
    image = pipe(
        prompt_schedule,
        temperature = temperature,
        num_inference_steps=num_inference_steps,
        callback=save_intermediate,
        callback_steps=1
    ).images[0]

    # Convert PIL Image to numpy array
    image_np = np.array(image)

    # Plot random samples
    num_samples = 5
    random_indices = [step_size, step_size*2, step_size*3, step_size*4, step_size*5]
    random_indices.sort()

    plt.figure(figsize=(20, 4))
    for i, idx in enumerate(random_indices):
        plt.subplot(1, num_samples +1, i + 1)
        plt.imshow(intermediate_images[idx])
        plt.title(f"After prompt: {i} Step {idx}")
        plt.axis('off')

    # Plot final image
    plt.subplot(1, num_samples+1, num_samples+1)
    plt.imshow(image_np)
    plt.title("Final Image")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

In [59]:
import torch
from scope_diffuser import SCoPEDiffusionPipeline
import matplotlib.pyplot as plt
import random
import numpy as np

temperatures = [0.5,1,5,10,100]
num_inference_steps = 200
step_size = 10
plt.figure(figsize=(20, 8))
# Load the model
model_id = "stabilityai/stable-diffusion-2-1"
pipe1 = SCoPEDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True)
pipe1 = pipe1.to("cuda:1")

prompt_schedule = [
    (0, "A photographer in a bustling city."),  
    (step_size, "A photographer capturing a candid moment in a vibrant city street, camera poised."),  
    (step_size * 2, "A photographer capturing a fleeting moment in a vibrant city street, surrounded by colorful buildings adorned with murals."),  
    (step_size * 3, "A photographer capturing a fleeting moment in a vibrant city street, surrounded by colorful buildings, bustling crowds, and street vendors selling goods."),  
    (step_size * 4, "A photographer capturing a fleeting moment in a vibrant city street, surrounded by colorful buildings, bustling crowds, and bright neon signs illuminating the night."),  
]


for temp_id, temperature in enumerate(temperatures):

    torch.manual_seed(42)
    image = pipe1(
        prompt_schedule,
        temperature = temperature,
        num_inference_steps=num_inference_steps,
        callback=None,
        callback_steps=1
    ).images[0]

    # Convert PIL Image to numpy array
    image_np = np.array(image)


    plt.subplot(1, len(temperatures) + 1, temp_id + 1)
    plt.imshow(image_np)
    plt.title(f"{temperature}")
    plt.axis('off')


from diffusers import StableDiffusionPipeline

# Load the model
pipe2 = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True)
pipe2 = pipe2.to("cuda:1")
torch.manual_seed(42)
image = pipe2(
    prompt_schedule[-1][1],
    num_inference_steps=num_inference_steps,
    callback=None,
    callback_steps=1
).images[0]
image_np = np.array(image)
# Plot final image
plt.subplot(1, len(temperatures)+1, len(temperatures)+1)
plt.imshow(image_np)
plt.title("Stable Diffusion")
plt.axis('off')
plt.suptitle(f"prompt: {prompt_schedule[-1][1]}\nstep size = {step_size}",fontsize=16, y=0.85)
plt.tight_layout()
plt.show()
plt.savefig(f'/home/ketanss/files/scope/scope_generations/{prompt_schedule[-1][1]}.png')

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]